# Gold App — painel do funcionário

Esta Gold atende o painel do funcionário com três tabelas:

- `gold_app_funcionario_fila`: uma linha por alerta aberto e atribuído a um operador.
- `gold_app_funcionario_kpis`: uma linha por operador, incluindo as contagens dos cartões.
- `gold_app_funcionario_leituras`: leituras associadas aos alertas para o gráfico da ocorrência.

## Regra provisória de prazo

A vida útil de referência começa na abertura do alerta. Até confirmação do grupo, 40% e 70% são pontos de escalonamento, “próximo do vencimento” começa em 70%, e o SLA vence ao atingir 100%. A regra fica centralizada nos parâmetros da primeira célula de código. Alertas sem vida útil de referência ficam como `SEM_REFERENCIA` e não entram nas contagens de vencimento.

A gravidade é padronizada para `BAIXO`, `ATENCAO` e `CRITICO`, removendo acentos e variações de gênero; o valor original permanece na tabela. “Resolvidos hoje” fica disponível na Gold para relatórios e pode ficar fora da tela inicial do app.

A fila considera alertas abertos atribuídos ao funcionário. O painel deve filtrar as tabelas pelo `id_usuario` do funcionário e ordenar a fila por `prioridade_fila`, depois pelo maior percentual consumido do prazo para itens vencidos/próximos, e por data de abertura para os demais.


In [ ]:
import re

from pyspark.sql import DataFrame, Window
from pyspark.sql import functions as F

spark.conf.set("spark.sql.session.timeZone", "America/Sao_Paulo")

dbutils.widgets.text("catalogo_destino", "workspace")
dbutils.widgets.text("schema_silver", "silver_app")
dbutils.widgets.text("schema_gold", "gold_app")
dbutils.widgets.dropdown(
    "tabela_amostra", "gold_app_funcionario_fila",
    ["gold_app_funcionario_fila", "gold_app_funcionario_kpis", "gold_app_funcionario_leituras"],
)
CATALOGO = dbutils.widgets.get("catalogo_destino").strip()
SCHEMA_SILVER = dbutils.widgets.get("schema_silver").strip()
SCHEMA_GOLD = dbutils.widgets.get("schema_gold").strip()
TABELA_AMOSTRA = dbutils.widgets.get("tabela_amostra").strip()

# Hipótese provisória, revisar quando o responsável pela regra de SLA retornar.
PCT_ESCALONAMENTO_GESTOR = 0.40
PCT_ESCALONAMENTO_ADMIN = 0.70
PCT_PROXIMO_VENCIMENTO = 0.70
PCT_SLA_VENCIDO = 1.00


def validar_identificador(valor: str, nome: str) -> str:
    if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", valor):
        raise ValueError(f"Identificador inválido em {nome}: {valor!r}")
    return valor


CATALOGO = validar_identificador(CATALOGO, "catalogo_destino")
SCHEMA_SILVER = validar_identificador(SCHEMA_SILVER, "schema_silver")
SCHEMA_GOLD = validar_identificador(SCHEMA_GOLD, "schema_gold")
SILVER = f"{CATALOGO}.{SCHEMA_SILVER}"
GOLD = f"{CATALOGO}.{SCHEMA_GOLD}"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOLD}")


def ler_silver(nome: str) -> DataFrame:
    tabela = f"{SILVER}.silver_app_{validar_identificador(nome, 'nome_tabela')}"
    if not spark.catalog.tableExists(tabela):
        raise ValueError(f"Tabela Silver não encontrada: {tabela}. Execute a Silver primeiro.")
    return spark.table(tabela)


def gravar_gold(df: DataFrame, nome: str) -> None:
    tabela = f"{GOLD}.{validar_identificador(nome, 'nome_tabela')}"
    (
        df.withColumn("_gold_built_at", F.current_timestamp())
        .write.format("delta").mode("overwrite")
        .option("overwriteSchema", "true").saveAsTable(tabela)
    )
    print(f"Gravada: {tabela}")


def intervalo_para_segundos(coluna):
    """Converte texto JDBC HH:MM[:SS] ou texto de unidade retornado por PGInterval."""
    valor = F.lower(F.trim(coluna.cast("string")))
    hhmm = valor.rlike(r"^-?\d+:\d{1,2}(:\d{1,2}(\.\d+)?)?$")
    horas_hhmm = F.regexp_extract(valor, r"^(-?\d+):", 1).cast("double")
    minutos_hhmm = F.regexp_extract(valor, r"^-?\d+:(\d{1,2})", 1).cast("double")
    segundos_hhmm = F.regexp_extract(valor, r"^-?\d+:\d{1,2}:(\d{1,2}(\.\d+)?)$", 1).cast("double")
    horas_txt = F.regexp_extract(valor, r"(-?\d+(?:\.\d+)?)\s*(?:hours?|hrs?)\b", 1).cast("double")
    minutos_txt = F.regexp_extract(valor, r"(-?\d+(?:\.\d+)?)\s*(?:minutes?|mins?)\b", 1).cast("double")
    segundos_txt = F.regexp_extract(valor, r"(-?\d+(?:\.\d+)?)\s*(?:seconds?|secs?)\b", 1).cast("double")
    horas = F.when(hhmm, horas_hhmm).otherwise(F.coalesce(horas_txt, F.lit(0.0)))
    minutos = F.when(hhmm, minutos_hhmm).otherwise(F.coalesce(minutos_txt, F.lit(0.0)))
    segundos = F.when(hhmm, F.coalesce(segundos_hhmm, F.lit(0.0))).otherwise(F.coalesce(segundos_txt, F.lit(0.0)))
    reconhecido = valor.isNotNull() & (hhmm | valor.rlike(r"hours?|hrs?|minutes?|mins?|seconds?|secs?"))
    return F.when(reconhecido, horas * 3600 + minutos * 60 + segundos)


alerta = ler_silver("alerta")
atendimento = ler_silver("atendimento")
usuario = ler_silver("usuario")
camara = ler_silver("estado_camara_atual")
leitura = ler_silver("leitura_temperatura")


In [ ]:
# Se houver mais de um atendimento associado ao alerta, seleciona o registro mais recente.
janela_atendimento = Window.partitionBy("id_alerta").orderBy(
    F.col("data_hora_reconhecimento").desc_nulls_last(), F.col("id_atendimento").desc()
)
atendimento_atual = (
    atendimento.withColumn("_ordem", F.row_number().over(janela_atendimento))
    .filter(F.col("_ordem") == 1).drop("_ordem")
)

# A junção mantém o grão de uma linha por alerta.
base = (
    alerta.alias("a")
    .join(atendimento_atual.alias("t"), "id_alerta", "left")
    .join(
        usuario.select(
            F.col("id_usuario").alias("_id_usuario"),
            F.col("nome_usuario").alias("nome_funcionario"),
            F.col("nivel_acesso").alias("nivel_acesso_funcionario"),
        ).alias("u"),
        F.col("t.id_usuario_responsavel") == F.col("u._id_usuario"), "left",
    )
    .join(camara.alias("c"), F.col("a.id_camara") == F.col("c.id_camara"), "left")
    .select(
        F.col("a.id_alerta"), F.col("t.id_atendimento"),
        F.col("t.id_usuario_responsavel").alias("id_usuario"),
        F.col("u.nome_funcionario"), F.col("u.nivel_acesso_funcionario"),
        F.col("a.id_camara"), F.col("c.modelo_camara"), F.col("c.localizacao"),
        F.col("c.temperatura_atual"), F.col("c.temperatura_min"), F.col("c.temperatura_max"),
        F.col("c.ultima_leitura_em"), F.col("c.estado_conectividade"),
        F.col("a.data_hora_alerta"), F.col("a.vida_util_referencia_horas"),
        F.col("a.nivel_atual"), F.col("a.nivel_gravidade"), F.col("a.status_alerta"),
        F.col("t.status_atendimento"), F.col("t.data_hora_reconhecimento"),
        F.col("t.data_hora_resolucao"),
    )
)

# Mantém o rótulo original e cria uma chave canônica sem acentos.
chave_gravidade = F.translate(
    F.upper(F.trim(F.col("nivel_gravidade"))),
    "ÁÀÂÃÄÉÈÊËÍÌÎÏÓÒÔÕÖÚÙÛÜÇ", "AAAAAEEEEIIIIOOOOOUUUUC",
)
base = base.withColumn("_gravidade_chave", chave_gravidade).withColumn(
    "gravidade_padronizada",
    F.when(F.col("_gravidade_chave").isin("BAIXO", "BAIXA"), "BAIXO")
    .when(F.col("_gravidade_chave") == "ATENCAO", "ATENCAO")
    .when(F.col("_gravidade_chave").isin("CRITICO", "CRITICA"), "CRITICO")
    .otherwise("OUTRA"),
)

# O relógio começa na abertura do alerta; para alertas resolvidos, para na resolução.
fim_contagem = F.coalesce(F.col("data_hora_resolucao"), F.current_timestamp())
base = (
    base
    .withColumn("vida_util_referencia_segundos", intervalo_para_segundos(F.col("vida_util_referencia_horas")))
    .withColumn("tempo_decorrido_segundos", F.greatest(
        F.lit(0), F.unix_timestamp(fim_contagem) - F.unix_timestamp("data_hora_alerta")
    ))
    .withColumn(
        "percentual_consumido_sla",
        F.when(F.col("vida_util_referencia_segundos") > 0,
               F.round(F.col("tempo_decorrido_segundos") / F.col("vida_util_referencia_segundos"), 4)),
    )
    .withColumn(
        "minutos_restantes_sla",
        F.when(F.col("vida_util_referencia_segundos") > 0,
               F.ceil((F.col("vida_util_referencia_segundos") - F.col("tempo_decorrido_segundos")) / 60)),
    )
    .withColumn(
        "estado_sla",
        F.when(F.col("vida_util_referencia_segundos").isNull() | (F.col("vida_util_referencia_segundos") <= 0), "SEM_REFERENCIA")
        .when(F.col("percentual_consumido_sla") >= F.lit(PCT_SLA_VENCIDO), "SLA_VENCIDO")
        .when(F.col("percentual_consumido_sla") >= F.lit(PCT_PROXIMO_VENCIMENTO), "PROXIMO_VENCIMENTO")
        .otherwise("NO_PRAZO"),
    )
    .withColumn(
        "em_aberto",
        F.col("status_alerta").isin("ATIVO", "RECONHECIDO")
        & F.col("status_atendimento").isin("PENDENTE", "EM_ANDAMENTO"),
    )
    .withColumn("resolvido_hoje", F.to_date("data_hora_resolucao") == F.current_date())
    .withColumn(
        "prioridade_fila",
        F.when(F.col("gravidade_padronizada") == "CRITICO", 1)
        .when(F.col("estado_sla") == "SLA_VENCIDO", 2)
        .when(F.col("estado_sla") == "PROXIMO_VENCIMENTO", 3)
        .when(F.col("status_atendimento") == "EM_ANDAMENTO", 4)
        .otherwise(5),
    )
    .withColumn(
        "descricao_prioridade",
        F.when(F.col("prioridade_fila") == 1, "CRITICO")
        .when(F.col("prioridade_fila") == 2, "SLA_VENCIDO")
        .when(F.col("prioridade_fila") == 3, "PROXIMO_VENCIMENTO")
        .when(F.col("prioridade_fila") == 4, "EM_ATENDIMENTO")
        .otherwise("PENDENTE"),
    )
    .drop("_gravidade_chave")
)

fila = (
    base.filter(
        F.col("em_aberto") & F.col("id_usuario").isNotNull()
        & (F.col("nivel_acesso_funcionario") == "OPERADOR")
    )
    .withColumn(
        "ordem_dentro_prioridade",
        F.when(F.col("prioridade_fila").isin(2, 3), -F.coalesce(F.col("percentual_consumido_sla"), F.lit(0.0)))
        .otherwise(F.unix_timestamp("data_hora_alerta")),
    )
    # O consumidor deve ordenar por prioridade_fila e ordem_dentro_prioridade.
    .drop("em_aberto", "resolvido_hoje")
)


In [ ]:
# Uma linha por operador, incluindo quem não tem alertas atribuídos.
operadores = usuario.filter(F.col("nivel_acesso") == "OPERADOR").select(
    "id_usuario", F.col("nome_usuario").alias("nome_funcionario"), "id_cd"
)
contagens = (
    base.filter(F.col("id_usuario").isNotNull()).groupBy("id_usuario").agg(
        F.sum(F.when(F.col("em_aberto"), 1).otherwise(0)).alias("alertas_atribuidos_a_mim"),
        F.sum(F.when(F.col("em_aberto") & (F.col("gravidade_padronizada") == "CRITICO"), 1).otherwise(0)).alias("alertas_criticos"),
        F.sum(F.when(F.col("em_aberto") & (F.col("status_atendimento") == "EM_ANDAMENTO"), 1).otherwise(0)).alias("alertas_em_atendimento"),
        F.sum(F.when(F.col("em_aberto") & (F.col("estado_sla") == "PROXIMO_VENCIMENTO"), 1).otherwise(0)).alias("proximos_do_vencimento"),
        F.sum(F.when(F.col("em_aberto") & (F.col("estado_sla") == "SLA_VENCIDO"), 1).otherwise(0)).alias("sla_vencido"),
        F.sum(F.when(F.col("resolvido_hoje"), 1).otherwise(0)).alias("resolvidos_hoje"),
    )
)
kpis = (
    operadores.join(contagens, "id_usuario", "left")
    .fillna(0, subset=[
        "alertas_atribuidos_a_mim", "alertas_criticos", "alertas_em_atendimento",
        "proximos_do_vencimento", "sla_vencido", "resolvidos_hoje",
    ])
)

# Uma linha por leitura vinculada a um alerta, preservando o histórico para o gráfico.
leituras_funcionario = (
    leitura.filter(F.col("id_alerta").isNotNull()).alias("l")
    .join(
        base.select(
            "id_alerta", "id_usuario", "id_camara", "data_hora_alerta",
            "temperatura_min", "temperatura_max",
        ).alias("b"),
        F.col("l.id_alerta") == F.col("b.id_alerta"), "inner",
    )
    .select(
        F.col("l.id_alerta"), F.col("b.id_usuario"), F.col("b.id_camara"),
        F.col("l.id_leitura"), F.col("l.data_hora_leitura"), F.col("l.temperatura"),
        F.col("b.temperatura_min"), F.col("b.temperatura_max"),
        (F.col("l.data_hora_leitura") >= F.col("b.data_hora_alerta")).alias("leitura_apos_inicio_alerta"),
        ((F.col("l.temperatura") < F.col("b.temperatura_min"))
         | (F.col("l.temperatura") > F.col("b.temperatura_max"))).alias("temperatura_fora_da_faixa"),
    )
)


In [ ]:
TABELAS_GOLD = {
    "gold_app_funcionario_fila": fila,
    "gold_app_funcionario_kpis": kpis,
    "gold_app_funcionario_leituras": leituras_funcionario,
}
for nome, df in TABELAS_GOLD.items():
    gravar_gold(df, nome)
print(f"Gold do funcionário concluída: {len(TABELAS_GOLD)} tabelas em {GOLD}")


In [ ]:
TABELAS_AMOSTRA = set(TABELAS_GOLD)
if TABELA_AMOSTRA not in TABELAS_AMOSTRA:
    raise ValueError(f"tabela_amostra inválida: {TABELA_AMOSTRA}")
tabela = f"{GOLD}.{TABELA_AMOSTRA}"
df_amostra = spark.table(tabela)
print(f"Amostra: {tabela}")
df_amostra.printSchema()
display(df_amostra.limit(20))
